In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv

from camel.agents.chat_agent import ChatAgent
from camel.configs.openai_config import ChatGPTConfig
from camel.messages.base import BaseMessage
from camel.models import ModelFactory
from camel.types import ModelPlatformType, ModelType
from camel.societies.workforce import Workforce
from camel.toolkits import FunctionTool
from camel.tasks.task import Task

import nest_asyncio

In [ ]:
load_dotenv()
nest_asyncio.apply()
os.environ["OPENAI_API_KEY"] = "your-api-key-here"

In [ ]:
PLANNER_PROMPT = "Planner prompt goes here"
RESEARCHER_PROMPT = "Researcher prompt goes here"
REPORTER_PROMPT = "Reporter prompt goes here"
JUDGE_PROMPT = "Judge prompt goes here"

In [ ]:
def read_paper_text():
    with open("path/to/paper", "r") as file:
        return file.read()

def read_eagle_guidelines():
    with open("path/to/guidelines", "r") as file:
        return file.read()

paper_tools = FunctionTool(read_paper_text)
eagle_guidelines_tool = FunctionTool(read_eagle_guidelines)

In [ ]:
planner_agent = ChatAgent(
    system_message=BaseMessage.make_assistant_message(
        role_name="Planner Agent",
        content=PLANNER_PROMPT,
    ),
    model=ModelFactory.create(
        model_platform=ModelPlatformType.OPENAI,
        model_type=ModelType.O3_MINI,
        model_config_dict=ChatGPTConfig().as_dict(),
    ),
    tools=[paper_tools],
)

researcher_agent = ChatAgent(
    system_message=BaseMessage.make_assistant_message(
        role_name="Researcher Agent",
        content=RESEARCHER_PROMPT,
    ),
    model=ModelFactory.create(
        model_platform=ModelPlatformType.OPENAI,
        model_type=ModelType.GPT_4O,
        model_config_dict=ChatGPTConfig().as_dict(),
    ),
    tools=[paper_tools],
)

In [ ]:
reporter_agent = ChatAgent(
    system_message=BaseMessage.make_assistant_message(
        role_name="Reporter Agent",
        content=REPORTER_PROMPT,
    ),
    model=ModelFactory.create(
        model_platform=ModelPlatformType.OPENAI,
        model_type=ModelType.GPT_4O,
        model_config_dict=ChatGPTConfig().as_dict(),
    ),
)

judge_agent = ChatAgent(
    system_message=BaseMessage.make_assistant_message(
        role_name="Judge Agent",
        content=JUDGE_PROMPT,
    ),
    model=ModelFactory.create(
        model_platform=ModelPlatformType.OPENAI,
        model_type=ModelType.O3_MINI,
        model_config_dict=ChatGPTConfig().as_dict(),
    ),
)

In [ ]:
workforce = Workforce("EAGLE Case Extraction Workforce")

workforce.add_single_agent_worker(
    "Planner Agent",
    worker=planner_agent,
).add_single_agent_worker(
    "Researcher Agent",
    worker=researcher_agent,
).add_single_agent_worker(
    "Reporter Agent",
    worker=reporter_agent,
).add_single_agent_worker(
    "Judge Agent",
    worker=judge_agent,
)